# Session 5: Simulated annealing — acceptance and exploration

Use the Week 3 page in ELO (Session 5 section), its worksheet S5.1–S5.5,
and the [notation glossary](NotationGlossary.pdf).
This separate short notebook leaves Session 3's route intact. The full Rastrigin
implementation and comparison in notebook 01 belong to Session 7.

**Weekly timing:** Sessions 5 and 6 take place on the same day, with a
15-minute rest break. No homework or setup is required during that gap.
Complete this follow-up after both sessions and before Session 7 the next week.

Follow-up: 30 minutes reading and 45 minutes practice (10 worksheet, 25 this
notebook, 10 discussion and your own explanation). O2 is covered once by S5.1–S5.2.
The 25-minute route is **A: 6, B: 4, C: 5, D: 5, E: 5 minutes**.
Use A's supplied transition tables as references for your predictions.
Reuse the worksheet calculations: S5.3 and E count as the same work.
No project experiment is assigned. Predict first; run A–E; change one
specified setting at a time.

## A. Separate proposal, conditional acceptance, and actual transition

Use $D=\{0,1,2,3,4\}$ and $f:D\to\mathbb R$ with
$(f(0),\ldots,f(4))=(0,3,1,4,2)$. Fix step length $h=1$.
All random variables share a probability space $(\Omega,\mathcal F,\Pr)$.
For a fixed current state $x\in D$, draw $S\in\{-1,1\}$ with probabilities
$1/2,1/2$. The **raw proposal** is $Y_x^{\mathrm{raw}}=x+hS$.
The **tentative state** $\widetilde Y_x$ equals the raw proposal if it is
in $D$, and equals $x$ otherwise. An invalid raw proposal thus stays at
$x$; it is neither reflected nor redrawn, and its objective is not evaluated.

**Proposal law.** Define $q(y\mid x)=\Pr(\widetilde Y_x=y)$ for $x,y\in D$.
With current states as rows and tentative states as columns, both ordered
$0,1,2,3,4$, the matrix $Q_{xy}=q(y\mid x)$ is
$$Q=\frac12\begin{pmatrix}
1&1&0&0&0\\1&0&1&0&0\\0&1&0&1&0\\0&0&1&0&1\\0&0&0&1&1
\end{pmatrix}.$$
Its rows are probability vectors, and it is symmetric: $q(y\mid x)=q(x\mid y)$.
The diagonal entries at 0 and 4 represent invalid raw proposals handled by staying.

**Conditional acceptance.** For feasible $x,y\in D$ and temperature $T>0$,
write $\Delta(x,y)=f(y)-f(x)$ and define
$$\alpha_T(x,y)=\min\{1,e^{-\Delta(x,y)/T}\}.$$
Given tentative state $y$, compare an independent $U\sim\mathrm{Uniform}(0,1)$
with this value: accept if $U<\alpha_T(x,y)$, otherwise retain $x$.
For a realization $\omega\in\Omega$, $u=U(\omega)$ determines one decision.
The conditional-probability interpretation applies when $q(y\mid x)>0$.
The self-proposal has $\alpha_T(x,x)=1$, hence leaves the state unchanged.
In the algorithm, an invalid raw proposal is still recorded as boundary
rejection, not an accepted move. Equality has probability zero in the ideal
continuous model; NumPy uses values in `[0, 1)` and the strict comparison.

**Actual transition.** Let $X_x^+$ be the state after both stages and define
$K_T(x,y)=\Pr(X_x^+=y)$. Then
$$K_T(x,y)=q(y\mid x)\alpha_T(x,y)\quad(x\ne y),$$
$$K_T(x,x)=q(x\mid x)+\sum_{z\in D,\ z\ne x}q(z\mid x)[1-\alpha_T(x,z)]
=1-\sum_{z\in D,\ z\ne x}K_T(x,z).$$
There are two ways to stay: a tentative self-proposal or rejection of a
proposal to another state. Symmetry of $Q$ does not imply symmetry of $K_T$.

The product uses **conditional multiplication, not independence of proposal
and acceptance events**. For fixed current state $x$, let $B$ be the event
that the tentative proposal is accepted. When $q(y\mid x)>0$ and $y\ne x$,
$$\Pr(\widetilde Y_x=y, B)
=\Pr(\widetilde Y_x=y)\Pr(B\mid\widetilde Y_x=y)
=q(y\mid x)\alpha_T(x,y).$$
If $q(y\mid x)=0$, both the move probability and the product are zero.
The independent uniform input creates the required conditional acceptance
probability; the acceptance event itself depends on the proposed state.
For staying, add the **mutually exclusive** events: tentative state $x$, or
some tentative state $z\ne x$ followed by rejection. This explains the
sum in $K_T(x,x)$: multiply within each case, then add across cases.

**Predict before reading the worked table, as in S5.1:** at $x=4$, $T=2$,
assign probabilities to the three mechanisms below and group them by actual
next state. Identify which part of the stay probability is $q(4\mid4)$.
Given raw proposal 3, decide for $u=0.2$ and $u=0.8$ using **alpha**, not K.
Use this same reasoning for S5.1 and the notebook; count the work once.

**Worked example: three mechanisms, two possible next states.** The raw
proposals 3 and 5 each have probability $1/2$. Moving to 3 raises the
objective by $f(3)-f(4)=4-2=2$, so acceptance **given that proposal** has
probability $\alpha_2(4,3)=e^{-1}\approx0.367879$.

| Update mechanism from state 4 | Probability | Actual next state |
| --- | ---: | ---: |
| Raw 5 is invalid: stay | $\tfrac12=0.5$ | 4 |
| Raw 3 is accepted: $U<e^{-1}$ | $\tfrac12e^{-1}\approx0.183940$ | 3 |
| Raw 3 is rejected: $U\geq e^{-1}$ | $\tfrac12(1-e^{-1})\approx0.316060$ | 4 |

These mechanisms are **disjoint and exhaustive**: exactly one occurs in
each update, and their probabilities sum to 1. Grouping rows by actual
next state gives
$$K_2(4,3)=\tfrac12e^{-1}\approx0.183940,$$
$$K_2(4,4)=\underbrace{\tfrac12}_{\text{invalid raw 5}}
+\underbrace{\tfrac12(1-e^{-1})}_{\text{raw 3 rejected}}
\approx0.816060.$$
Thus $q(4\mid4)=0.5$ is only the tentative self-proposal contribution to
staying. Rejection of the feasible proposal adds approximately $0.316060$.
Neither $f(5)$ nor $\alpha_2(4,5)$ is defined or evaluated.

In B, the first fixed trace row ($x_0=4$, raw proposal 3, $T=2$, $u=0.2$)
realizes the accepted branch. The three rows above are possible mechanisms,
not three transitions that all occur. An invalid raw proposal adds no
objective call; proposing 3 adds one whether accepted or rejected.
Initialization is counted separately. The table is an exact reference,
not a simulated frequency estimate or part of that objective-call ledger.

The supplied code also compares $T=0.5$ with $T=2$; it is not another
simulation task. Displayed decimals are rounded evaluations of exact formulas.

The existing curves show **conditional acceptance** for a worsening change
$\Delta>0$, not the probability of proposing or actually moving. For $4\to3$,
$\Delta=2$ and $q(3\mid4)=1/2$, so actual move probabilities are half the
curve heights. Temperature has the same units as objective differences;
$\Delta/T$ is dimensionless. The formulas require $T>0$.

The exact matrix calculations use the supplied landscape, outside the
objective-call ledger in B/C. Session 6 develops the Markov-chain
interpretation of these complete transitions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

energy = np.array([0., 3., 1., 4., 2.])
h = 1

def acceptance(delta, temperature):
    if not np.isfinite(temperature) or temperature <= 0:
        raise ValueError('Temperature must be finite and positive.')
    return np.exp(-np.maximum(np.asarray(delta, dtype=float), 0)/temperature)

for temperature in (0.5, 2.0):
    print('T =', temperature, 'probabilities:', acceptance([-2, 0, 2], temperature))
assert np.allclose(acceptance([-2, 0, 2], 2), [1, 1, np.exp(-1)])

# Exact reference from the known landscape: no simulation or objective-call ledger.
Q = np.zeros((len(energy), len(energy)))
for x_reference in range(len(energy)):
    for sign_reference in (-1, 1):
        raw_reference = x_reference + h*sign_reference
        tentative_reference = raw_reference if raw_reference in range(len(energy)) else x_reference
        Q[x_reference, tentative_reference] += 0.5

def transition_matrix(temperature):
    # Row x, column y: delta = f(y) - f(x), for feasible states only.
    alpha = acceptance(energy[None, :] - energy[:, None], temperature)
    kernel = Q*alpha
    np.fill_diagonal(kernel, 0.0)
    # All rejected off-diagonal mass returns to the current state.
    np.fill_diagonal(kernel, 1.0-kernel.sum(axis=1))
    return kernel

assert np.allclose(Q.sum(axis=1), 1) and np.allclose(Q, Q.T)
assert np.allclose(Q[0], [0.5, 0.5, 0, 0, 0])
assert np.allclose(Q[4], [0, 0, 0, 0.5, 0.5])
kernel_T2 = transition_matrix(2.0)
with np.printoptions(precision=6, suppress=True):
    print('Q: rows = current state; columns = tentative state (0,...,4)')
    print(Q)
    print('K at T=2: rows = current state; columns = actual next state (0,...,4)')
    print(kernel_T2)

print('Exact comparison at current state x=4 (not a simulated frequency)')
print(f"{'Quantity':<29} {'T=0.5':>10} {'T=2':>10}")
comparison = []
for temperature in (0.5, 2.0):
    kernel = transition_matrix(temperature)
    alpha43 = float(acceptance(energy[3]-energy[4], temperature))
    comparison.append([Q[4, 3], Q[4, 4], alpha43, kernel[4, 3], kernel[4, 4], kernel[3, 4]])
    alpha = acceptance(energy[None, :] - energy[:, None], temperature)
    diagonal_by_causes = np.diag(Q) + (Q*(1-alpha)).sum(axis=1)
    assert np.all(kernel >= 0) and np.allclose(kernel.sum(axis=1), 1)
    assert np.allclose(np.diag(kernel), diagonal_by_causes)
    assert np.isclose(kernel[4, 3], 0.5*np.exp(-2/temperature))
    assert np.isclose(kernel[4, 4], 0.5+0.5*(1-np.exp(-2/temperature)))
    assert not np.allclose(kernel, kernel.T)
for label, low, high in zip(
        ('q(3|4)', 'q(4|4): boundary self-loop', 'alpha_T(4,3)',
         'K_T(4,3): actually move', 'K_T(4,4): remain at 4', 'K_T(3,4): reverse move'),
        comparison[0], comparison[1]):
    print(f'{label:<29} {low:10.6f} {high:10.6f}')
print('Given raw proposal 3 at T=2, compare u with alpha =', float(acceptance(2, 2)))
for fixed_u in (0.2, 0.8):
    print('u =', fixed_u, ':', 'accept' if fixed_u < acceptance(2, 2) else 'reject')
delta_grid = np.linspace(0, 5, 201)
fig_acceptance, ax = plt.subplots(figsize=(7, 3.5))
for temperature in (0.5, 1.0, 2.0):
    ax.plot(delta_grid, acceptance(delta_grid, temperature), label=f'T={temperature}')
ax.set(xlabel='Worsening objective change delta', ylabel='Acceptance probability', ylim=(0, 1.03))
ax.legend()
fig_acceptance.tight_layout()
plt.show()

## B. Trace current and retained values

Use $D=\{0,1,2,3,4\}$ with $(f(0),\ldots,f(4))=(0,3,1,4,2)$.
The unique global minimizer is $x^*=0$, with $f_*=0$. The points 2 and 4 are
local minima **relative to nearest-neighbor moves** (every feasible adjacent
point has larger value). This is a graph neighborhood, not the Euclidean
definition of local minimum on a finite set, under which every point is local.

Use the same $h=1$ proposal rule and Q from A. At transition $t$, distinguish
the supplied **raw proposal** $y_t$, its **boundary-handled tentative state**
(equal to $y_t$ if feasible, otherwise $x_{t-1}$), and the **actual next state**
$x_t$ after the acceptance decision. Rejection repeats the previous state.
Redrawing until feasible would give a different, asymmetric proposal law.

Let $\mathbb N_0=\{0,1,\ldots\}$. The initial random state is
$X_0:\Omega\to D$. After transition $t\geq1$, the state is $X_t:\Omega\to D$.
Define $M_t=\min_{0\leq j\leq t}f(X_j)$ for $t\in\mathbb N_0$.
In a realized trace $x_t=X_t(\omega)$ and $b_t=M_t(\omega)$.
A retained best state is a visited state whose objective equals $M_t$;
retain the first visited state in a tie. The code reports its objective
value, rather than its state label. Only $b_t$ must be non-increasing.
Index zero records initialization;
transition 1 produces $X_1$. Session 3 indexed its history by objective calls.

The graph's horizontal axis records completed transitions, including repeats
after rejection. Its vertical axis records objective values, not state labels.
Read the staircase at integer times; its connecting segments do not add transitions.

From $x_0=4$, trace the fixed proposals, temperatures, and uniform values below.
They are a hand calculation, not independent repetitions or a random sample.
Predict each decision before execution, concentrating on rows 3--6 as in S5.4.
The first output separates the raw proposal, tentative state, and actual state.
The second shows the probabilities at the **previous** state and the given temperature.
Here `q` is the probability of the tentative state, `alpha(raw)` is defined
only for a feasible raw proposal, and `K(prev,next)` is the probability of
the recorded next state **given the previous state**, before fixing a raw
proposal or uniform value. K accounts for all possible signs and uniforms;
it is not a decision threshold for this fixed row.

In the last row, raw proposal $-1$ is invalid and the tentative and actual
states are both 0. `alpha(raw)` displays **N/A**: neither $f(-1)$ nor
$\alpha_T(0,-1)$ is defined. The code retains zero internally only as a
boundary-rejection sentinel and never indexes `energy[-1]` for this proposal.
The mathematical folded self-proposal has $\alpha_T(0,0)=1$, but the
algorithm records **boundary rejection**, with no accepted move and no objective call.

The objective-call counter counts initialization and every feasible proposal,
including rejected or repeated points. The current value is reused without
an additional call. Exact Q/K diagnostics use the known reference landscape;
these explanatory calculations are outside that algorithm cost ledger.

In [ ]:
proposals = [3, 2, 1, 1, 0, -1]
temperatures = [2., 2., 1., 0.5, 0.5, 0.5]
uniforms = [0.2, 0.9, 0.8, 0.01, 0.4, 0.3]
x, best, evaluations = 4, energy[4], 1
trace = []
trace_diagnostics = []
for t, (y, temperature, u) in enumerate(zip(proposals, temperatures, uniforms), 1):
    previous = x
    feasible = 0 <= y < len(energy)
    probability = float(acceptance(energy[y]-energy[x], temperature)) if feasible else 0.0
    tentative = y if feasible else previous
    evaluations += int(feasible)
    accepted = feasible and u < probability
    if accepted:
        x = y
    best = min(best, energy[x])
    trace.append((t, y, temperature, u, probability, accepted, x, energy[x], best, evaluations))
    trace_diagnostics.append((previous, tentative, feasible, Q[previous, tentative],
                              transition_matrix(temperature)[previous, x]))
print('State and cost trace: raw -> tentative -> actual next state')
print('t previous raw tentative next f(next) best calls')
for row, detail in zip(trace, trace_diagnostics):
    print(f'{row[0]:1d} {detail[0]:8d} {row[1]:3d} {detail[1]:9d} '
          f'{row[6]:4d} {row[7]:7.1f} {row[8]:4.1f} {row[9]:5d}')
print('Probability diagnostics at the previous state (not empirical frequencies)')
print('t    T    u     q alpha(raw) K(prev,next) decision')
for row, detail in zip(trace, trace_diagnostics):
    raw_alpha = f'{row[4]:.6f}' if detail[2] else 'N/A'
    decision = 'boundary' if not detail[2] else ('accept' if row[5] else 'reject')
    print(f'{row[0]:1d} {row[2]:4.1f} {row[3]:4.2f} {detail[3]:5.2f} '
          f'{raw_alpha:>10} {detail[4]:12.6f} {decision}')
assert [row[6] for row in trace] == [3, 2, 2, 1, 0, 0]
assert [row[8] for row in trace] == [2, 1, 1, 1, 0, 0]
assert evaluations == 6  # Initial point plus five feasible proposals.

## C. Run one declared cooling schedule

For transition $t\in\{1,\ldots,N\}$ use $T_t=T_0\rho^{t-1}$, with
$T_0>0$, $0<\rho\leq1$, and $N=80$. Here $T_0$ names the initial schedule
parameter, so $T_1=T_0$. Every proposal advances the schedule, even an
infeasible one. This is **proposal-indexed**, unlike the evaluation-indexed
project schedule. The code uses $T_0=2$, $\rho=0.97$, and $x_0=4$.

The model uses mutually independent signs $S_t\in\{-1,1\}$ with probabilities
$1/2,1/2$ and uniforms $U_t\sim\mathrm{Uniform}(0,1)$ for $t=1,\ldots,80$,
independent of the initial state. Here that state is fixed: $X_0=4$.
Array entry `step` is transition $t=$ `step + 1`. Both input arrays have
80 entries; state/current/best histories have 81 entries including time zero.
The schedule has 80 temperatures, one per transition. Every step consumes the
corresponding sign and uniform, including boundary rejection.

A seeded pseudorandom generator approximates the model; it does not prove
independence. The displayed `accepted` count includes feasible accepted moves,
`boundary` counts invalid proposals, and `evaluations` counts objective calls.
Check `evaluations = 1 + proposals - boundary`.
The ratio `accepted / proposals` is one realized path statistic: its
numerator counts feasible accepted moves, and its denominator includes
boundary proposals. It is not q for a particular tentative state, alpha
for a particular proposal, or an entry of one fixed K. Current states and
temperatures change along this path. Predict whether current values or
retained best values can rise.

In [ ]:
def anneal_path(signs, uniforms, initial_temperature=2., cooling=0.97, start=4):
    if len(signs) != len(uniforms) or not 0 < cooling <= 1:
        raise ValueError('Use equally long input arrays and 0 < cooling <= 1.')
    if start not in range(len(energy)) or not np.all(np.isin(signs, [-1, 1])):
        raise ValueError('Invalid starting state or sign.')
    if np.any((uniforms < 0) | (uniforms >= 1)):
        raise ValueError('Uniform values must lie in [0,1).')
    acceptance(0, initial_temperature)  # Validate the temperature before running.
    x, calls, accepted, boundary = start, 1, 0, 0
    states, current, best, schedule = [x], [energy[x]], [energy[x]], []
    for step, (sign, u) in enumerate(zip(signs, uniforms)):
        temperature = initial_temperature*cooling**step
        probability = 0.0
        y = x + int(sign)
        if y in range(len(energy)):
            calls += 1
            probability = float(acceptance(energy[y]-energy[x], temperature))
            if u < probability:
                x = y
                accepted += 1
        else:
            boundary += 1
        schedule.append(temperature)
        states.append(x)
        current.append(energy[x])
        best.append(min(best[-1], current[-1]))
    return dict(states=np.array(states), current=np.array(current), best=np.array(best),
                temperatures=np.array(schedule), evaluations=calls, accepted=accepted,
                boundary=boundary, proposals=len(signs))

rng = np.random.default_rng(5050)
signs = rng.choice([-1, 1], size=80)
uniforms = rng.random(80)
baseline = anneal_path(signs, uniforms)
print({key: baseline[key] for key in ('proposals', 'evaluations', 'accepted', 'boundary')})
assert baseline['evaluations'] == 1+baseline['proposals']-baseline['boundary']
assert np.all(np.diff(baseline['best']) <= 0)
fig_path, axes = plt.subplots(1, 2, figsize=(10, 3.7))
axes[0].plot(np.arange(5), energy, 'o-')
axes[0].set(xlabel='State', ylabel='Objective f(x)', title='Only five feasible states', xticks=range(5))
axes[1].step(range(81), baseline['current'], where='post', label='Current value')
axes[1].step(range(81), baseline['best'], where='post', label='Retained best', linewidth=2)
axes[1].set(xlabel='Completed transitions', ylabel='Objective value', title='One finite annealing trajectory')
axes[1].legend()
fig_path.tight_layout()
plt.show()

## D. Make one controlled change and explain it

Preserve the baseline. Predict the effect of changing only `initial_temperature`
from 2 to 0.5. Reuse the same `signs` and `uniforms` in the cell below. These two
paths are **coupled**, not independent repetitions: identical random inputs
isolate a setting change, but once states differ, proposed locations differ too.
No theorem orders their final best values. A single pair of trajectories does
not give precise estimates of success probabilities or establish an optimal cooling schedule.

This finite geometric cooling rule is a heuristic. It is not a claim that
the global minimizer is reached. Changing temperature changes the complete
transition rule. Session 6 introduces Markov chains and invariant laws;
Session 7 returns to continuous annealing and fair repeated comparisons.

In [ ]:
changed = anneal_path(signs, uniforms, initial_temperature=0.5)
for label, result in [('Baseline T0=2', baseline), ('Changed T0=0.5', changed)]:
    print(label, 'final current =', result['current'][-1], 'final best =', result['best'][-1],
          'accepted =', result['accepted'], 'objective calls =', result['evaluations'])
fig_change, ax = plt.subplots(figsize=(7, 3.5))
for label, result in [('T0=2', baseline), ('T0=0.5', changed)]:
    ax.step(range(81), result['best'], where='post', label=label)
ax.set(xlabel='Completed transitions', ylabel='Retained best value', title='Same random inputs; one setting changed')
ax.legend()
fig_change.tight_layout()
plt.show()

## E. Continuous proposals: scale and temperature (5 minutes)

**Reuse S5.3; do not repeat the calculation as another assignment.**
Use $D_c=[-2,2]$ and $f_c:D_c\to\mathbb R$,
$f_c(x)=(x^2-1)^2+0.2x$. This continuous function attains a minimum on
its nonempty compact domain. We now use a different state space from A-D;
the continuous example starts at $x=1$, rather than the finite example's 4.
This section performs individual updates,
not a continuous annealing trajectory or a parameter sweep.

For current $x\in D_c$, scale $s>0$, and temperature $T>0$, the model draws
independent $Z\sim\mathcal N(0,1)$ and $U\sim\mathrm{Uniform}(0,1)$,
independent of earlier inputs, and proposes $Y=x+sZ$. For feasible $y$,
$$\Delta_c(x,y)=f_c(y)-f_c(x),\qquad
\alpha_T^c(x,y)=\min\{1,e^{-\Delta_c(x,y)/T}\}.$$
Accept when $U<\alpha_T^c(x,y)$; otherwise stay at $x$. An infeasible raw
proposal is rejected **before any candidate objective evaluation**.
The current objective is cached. Retain the smallest visited objective value.

**Fixed arithmetic, not random samples.** Each comparison restarts at
$x=1$, with supplied $z=-1$ and $u=0.2$. These rows are not successive
transitions or frequency estimates. The supplied continuous baseline is
$s=0.2,T=0.1$: $y=0.8$, $f_c(y)=0.2896$, $\Delta_c(1,0.8)=0.0896$, and
$\alpha_{0.1}^c(1,0.8)=e^{-0.896}\approx0.408199$, so the update accepts.
Use your worksheet answers to predict the rows that change **only** $T$
to $0.02$, and **only** $s$ to $0.1$. The starting value is $f_c(1)=0.2$
in every row. The helper is supplied; there is no implementation task.

Read two short output tables: candidate/threshold first, then actual
movement/cost. The raw displacement is $sz=y-x$; actual displacement is
$x^+-x$, where $x^+$ denotes the realized next state. Rejection makes the
latter zero. Candidate calls exclude the initial evaluation of $f_c(1)$.
The one-update retained best includes the initial and actual next states.
For the boundary example $x=1,s=1.5,z=1$, the raw candidate is 2.5:
reject for every $T>0$, with delta and acceptance **N/A**, not zero.

**Density reference.** The raw Gaussian proposal has density
$$q_s(y\mid x)=\frac{1}{s\sqrt{2\pi}}
\exp\!\left[-\frac{(y-x)^2}{2s^2}\right],\qquad y\in\mathbb R.$$
For each fixed real $y$, $\Pr(Y=y)=0$; density area over an interval gives
its probability. The density describes the raw proposal, not the actual
next-state law, which includes staying after rejection. Scale $s$ changes
the displacement distribution; $T$ changes acceptance for a fixed feasible
pair. The two panels show analytic functions, not empirical curves. The
left shows $y\in[0.4,2.1]$ and the upper boundary 2; the lower boundary
$-2$ is outside this view. Both raw densities have support $\mathbb R$.

In [ ]:
def f_c(value):
    if not np.isfinite(value) or not -2 <= value <= 2:
        raise ValueError('f_c is defined only on [-2, 2].')
    return (value**2-1)**2 + 0.2*value

def continuous_update(x, cached_value, scale, temperature, z, u, objective=f_c):
    # Caller supplies cached_value = f_c(x); do not evaluate the current point again.
    if not all(np.isfinite(v) for v in (x, cached_value, scale, temperature, z, u)):
        raise ValueError('All supplied arithmetic inputs must be finite.')
    if not -2 <= x <= 2 or scale <= 0 or temperature <= 0 or not 0 <= u < 1:
        raise ValueError('Require feasible x, positive s and T, and 0 <= u < 1.')
    raw = x + scale*z
    result = dict(raw=raw, candidate_value=None, delta=None, alpha=None,
                  decision='boundary', next_state=x, next_value=cached_value,
                  raw_displacement=scale*z, actual_displacement=0.0,
                  candidate_calls=0, retained_best=cached_value)
    if not -2 <= raw <= 2:
        return result  # No objective call at an infeasible raw proposal.
    candidate_value = objective(raw)
    delta = candidate_value-cached_value
    alpha = float(np.exp(-max(delta, 0.0)/temperature))
    accepted = u < alpha
    next_state = raw if accepted else x
    next_value = candidate_value if accepted else cached_value
    result.update(candidate_value=candidate_value, delta=delta, alpha=alpha,
                  decision='accept' if accepted else 'reject', next_state=next_state,
                  next_value=next_value, actual_displacement=next_state-x,
                  candidate_calls=1, retained_best=min(cached_value, next_value))
    return result

# Supplied one-update calculations: no random inputs are generated here.
x_c, z_c, u_c = 1.0, -1.0, 0.2
cached_c = f_c(x_c)  # Initial objective evaluation, separate from candidate calls.
continuous_settings = [('Continuous baseline', 0.2, 0.1, z_c),
                       ('Change only T', 0.2, 0.02, z_c),
                       ('Change only s', 0.1, 0.1, z_c),
                       ('Boundary example', 1.5, 0.1, 1.0)]
continuous_rows = [(case_name, continuous_update(x_c, cached_c, s_c, T_c, z_input, u_c))
                   for case_name, s_c, T_c, z_input in continuous_settings]
print('Fixed arithmetic inputs: each row starts again at x=1; u=0.2. Not frequencies.')
print('Candidate and threshold (N/A means outside the objective domain)')
print(f"{'Case':<21} {'raw y':>7} {'f_c(y)':>9} {'Delta_c':>9} {'alpha':>9}")
for case_name, row_c in continuous_rows:
    displayed = ['N/A' if row_c[key] is None else f'{row_c[key]:.6f}'
                 for key in ('candidate_value', 'delta', 'alpha')]
    print(f'{case_name:<21} {row_c["raw"]:7.3f} '
          + ' '.join(f'{value:>9}' for value in displayed))
print('Actual movement and candidate calls (initial objective evaluation excluded)')
print(f"{'Case':<21} {'decision':>8} {'next':>6} {'raw step':>9} {'actual':>9} {'calls':>6}")
for case_name, row_c in continuous_rows:
    print(f'{case_name:<21} {row_c["decision"]:>8} {row_c["next_state"]:6.3f} '
          f'{row_c["raw_displacement"]:9.3f} {row_c["actual_displacement"]:9.3f} '
          f'{row_c["candidate_calls"]:6d}')
print('Retained best after each separate update:', [row['retained_best'] for _, row in continuous_rows])

# Analytic references, not simulated or empirical curves.
fig_continuous, continuous_axes = plt.subplots(1, 2, figsize=(11, 3.8))
y_grid_c = np.linspace(0.4, 2.1, 401)
for scale_c in (0.1, 0.2):
    raw_density_c = np.exp(-0.5*((y_grid_c-x_c)/scale_c)**2)/(scale_c*np.sqrt(2*np.pi))
    continuous_axes[0].plot(y_grid_c, raw_density_c, label=f's={scale_c}')
continuous_axes[0].axvline(2, color='black', linestyle=':', label='Upper boundary 2')
continuous_axes[0].axvspan(2, 2.1, color='gray', alpha=0.15)
continuous_axes[0].set(xlabel='Raw candidate y', ylabel='Raw density q_s(y | 1)',
                       title='Scale changes the proposal density', xlim=(0.4, 2.1), ylim=(0, None))
continuous_axes[0].legend(fontsize=9)
delta_grid_c = np.linspace(0, 0.15, 301)
for temperature_c in (0.1, 0.02):
    curve_c, = continuous_axes[1].plot(delta_grid_c, np.exp(-delta_grid_c/temperature_c),
                                      label=f'T={temperature_c}')
    continuous_axes[1].plot(0.0896, np.exp(-0.0896/temperature_c), 'o', color=curve_c.get_color())
continuous_axes[1].axvline(0.0896, color='gray', linestyle=':', label='Given change 0.0896')
continuous_axes[1].set(xlabel='Feasible worsening change Delta_c', ylabel='Acceptance probability',
                       title='Temperature changes acceptance', xlim=(0, 0.15), ylim=(0, 1.03))
continuous_axes[1].legend(fontsize=9)
fig_continuous.tight_layout()
plt.show()

**Stop after A–E.** E reuses S5.3 and adds no written assignment.
In the same ten-minute explanation block, reuse S5.5 and write three sentences:
(1) at state 4 and $T=2$, compare $q(3\mid4)$, $\alpha_2(4,3)$, and
$K_2(4,3)$, and use invalid raw proposal 5 versus rejected raw proposal 3
to explain why $K_2(4,4)>q(4\mid4)$;
(2) for the continuous model, explain how $s$ changes the proposal distribution
while $T$ changes acceptance for a fixed feasible candidate, including how
rejection leaves the current state unchanged;
(3) distinguish the current value $f(x_t)$ from the retained best value
$b_t=\min_{0\leq j\leq t}f(x_j)$, and explain why one finite cooling run
alone does not certify optimality over unexamined feasible points (the toy
example's minimizer is known separately from its full objective table).
Count the worksheet and notebook explanation once. Optional sharing in the
course collaboration link announced in ELO
uses this same block. If setup is unavailable, use the supplied three-outcome
table, fixed trace, and continuous one-update calculation; label supplied
evidence explicitly. Do not run a parameter sweep or the later Rastrigin
comparison for this session.